# Grid based Dijkstra planning

---- 

- conda env : [ai_robotics](../../README.md#setup-a-conda-environment)

---

### Ref
- https://github.com/AtsushiSakai/PythonRobotics/
- https://github.com/AtsushiSakai/PythonRobotics/blob/master/PathPlanning/Dijkstra/dijkstra.py


### Imports and Configuration

In [1]:
import math
import heapq
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

show_animation = True

### Path Planner

In [2]:
class DijkstraPlanner:
    class Node:
        def __init__(self, x, y, cost, parent_index):
            self.x = x
            self.y = y
            self.cost = cost
            self.parent_index = parent_index

        def __lt__(self, other):
            return self.cost < other.cost  # for heapq priority comparison

    def __init__(self, ox, oy, resolution, robot_radius):
        self.resolution = resolution
        self.robot_radius = robot_radius
        self.calc_obstacle_map(ox, oy)
        self.motion = self.get_motion_model()

    # --------------------- Main Planning --------------------- #
    def planning(self, sx, sy, gx, gy):
        start_node = self.Node(self.calc_xy_index(sx, self.min_x),
                               self.calc_xy_index(sy, self.min_y), 0.0, -1)
        goal_node = self.Node(self.calc_xy_index(gx, self.min_x),
                              self.calc_xy_index(gy, self.min_y), 0.0, -1)

        open_heap = []  # (priority queue)
        heapq.heappush(open_heap, (start_node.cost, self.calc_index(start_node), start_node))
        closed_set = dict()

        path_steps = []  # record for animation
        open_dict = {self.calc_index(start_node): start_node}

        while open_heap:
            _, c_id, current = heapq.heappop(open_heap)
            if c_id in closed_set:
                continue

            closed_set[c_id] = current
            path_steps.append((self.calc_position(current.x, self.min_x),
                               self.calc_position(current.y, self.min_y)))

            if current.x == goal_node.x and current.y == goal_node.y:
                print("Goal found!")
                goal_node.parent_index = current.parent_index
                goal_node.cost = current.cost
                break

            # expand neighbors
            for move_x, move_y, move_cost in self.motion:
                node = self.Node(current.x + move_x,
                                 current.y + move_y,
                                 current.cost + move_cost, c_id)

                if not self.verify_node(node):
                    continue

                n_id = self.calc_index(node)
                if n_id in closed_set:
                    continue

                # Add or update node
                if n_id not in open_dict or open_dict[n_id].cost > node.cost:
                    open_dict[n_id] = node
                    heapq.heappush(open_heap, (node.cost, n_id, node))

        rx, ry = self.calc_final_path(goal_node, closed_set)
        return rx, ry, path_steps

    # --------------------- Helper Methods --------------------- #
    def calc_final_path(self, goal_node, closed_set):
        rx, ry = [self.calc_position(goal_node.x, self.min_x)], [
            self.calc_position(goal_node.y, self.min_y)]
        parent_index = goal_node.parent_index
        while parent_index != -1:
            n = closed_set[parent_index]
            rx.append(self.calc_position(n.x, self.min_x))
            ry.append(self.calc_position(n.y, self.min_y))
            parent_index = n.parent_index
        return rx[::-1], ry[::-1]

    def calc_position(self, index, minp):
        return index * self.resolution + minp

    def calc_xy_index(self, position, minp):
        return round((position - minp) / self.resolution)

    def calc_index(self, node):
        return (node.y - self.min_y) * self.x_width + (node.x - self.min_x)

    def verify_node(self, node):
        px = self.calc_position(node.x, self.min_x)
        py = self.calc_position(node.y, self.min_y)
        if px < self.min_x or py < self.min_y or px >= self.max_x or py >= self.max_y:
            return False
        if self.obstacle_map[node.x][node.y]:
            return False
        return True

    def calc_obstacle_map(self, ox, oy):
        self.min_x, self.min_y = round(min(ox)), round(min(oy))
        self.max_x, self.max_y = round(max(ox)), round(max(oy))
        self.x_width = round((self.max_x - self.min_x) / self.resolution)
        self.y_width = round((self.max_y - self.min_y) / self.resolution)

        self.obstacle_map = [[False for _ in range(self.y_width)]
                             for _ in range(self.x_width)]
        for ix in range(self.x_width):
            x = self.calc_position(ix, self.min_x)
            for iy in range(self.y_width):
                y = self.calc_position(iy, self.min_y)
                for iox, ioy in zip(ox, oy):
                    if math.hypot(iox - x, ioy - y) <= self.robot_radius:
                        self.obstacle_map[ix][iy] = True
                        break

    @staticmethod
    def get_motion_model():
        return [[1, 0, 1], [0, 1, 1], [-1, 0, 1], [0, -1, 1],
                [-1, -1, math.sqrt(2)], [-1, 1, math.sqrt(2)],
                [1, -1, math.sqrt(2)], [1, 1, math.sqrt(2)]]


In [3]:
def run_animation():
    sx, sy = -5.0, -5.0
    gx, gy = 50.0, 50.0
    grid_size = 2.0
    robot_radius = 1.0

    ox, oy = [], []
    for i in range(-10, 60):
        ox.append(float(i))
        oy.append(-10.0)
    for i in range(-10, 60):
        ox.append(60.0)
        oy.append(float(i))
    for i in range(-10, 61):
        ox.append(float(i))
        oy.append(60.0)
    for i in range(-10, 61):
        ox.append(-10.0)
        oy.append(float(i))
    for i in range(-10, 40):
        ox.append(20.0)
        oy.append(float(i))
    for i in range(0, 40):
        ox.append(40.0)
        oy.append(60.0 - i)

    planner = DijkstraPlanner(ox, oy, grid_size, robot_radius)
    rx, ry, path_steps = planner.planning(sx, sy, gx, gy)

    # Animation setup
    fig, ax = plt.subplots()
    ax.plot(ox, oy, ".k")
    ax.plot(sx, sy, "og", label="Start")
    ax.plot(gx, gy, "xb", label="Goal")
    line_search, = ax.plot([], [], "xc", alpha=0.5)
    line_path, = ax.plot([], [], "-r", linewidth=2, label="Path")
    ax.legend()
    ax.grid(True)
    ax.axis("equal")

    search_x, search_y = [], []

    def update(frame):
        if frame < len(path_steps):
            x, y = path_steps[frame]
            search_x.append(x)
            search_y.append(y)
            line_search.set_data(search_x, search_y)
        else:
            line_path.set_data(rx, ry)
        return line_search, line_path

    ani = FuncAnimation(fig, update, frames=len(path_steps) + 20,
                        interval=50, repeat=False)

    # # --- Inline HTML animation display ---
    # plt.close(fig)  # prevents duplicate static figure
    # return HTML(ani.to_html5_video())  # or ani.to_jshtml() if ffmpeg unavailable
    plt.close(fig)  # prevent duplicate static plot
    return ani


In [4]:
# Run main()
ani = run_animation()
HTML(ani.to_html5_video())

Goal found!
